# I3 — 요약 약점 개선: F1 base + I2 점수 조합 (학습 없음)

## 왜 하나

배포 판정기 F1 base는 **요약 과제에서 가장 약하다**(dev-L 8토큰 서비스 탐지율 0.156, X1 응답 AUROC 0.752 — HHEM보다도 낮음).
근거 전체를 읽는 I2는 요약을 0.156 → 0.266, QA를 0.408 → 0.492로 올렸지만 Data2txt가 0.662 → 0.580으로 내려가 전체로는 같았다(SSOT §5 I2, dev-L 보고값).
**두 판정기가 잘하는 과제가 다르므로 점수를 합치면** 요약은 I2 덕에 오르고 Data2txt는 F1 base 덕에 지킬 수 있다. 저장된 점수만 쓰므로 학습이 필요 없다.

## 무엇을 하나

1. F1 base 저장 점수(`final/scores_base.npy`)와 I2 저장 점수(`i2/scores_i2.npy`)를 같은 판정 지점 번호로 맞춘다(I2 노트북과 같은 함수).
2. 두 점수를 **cal-L 멀쩡한 응답의 8토큰 지점 점수** 평균·표준편차로 표준화한다(logit 크기가 판정기마다 달라서).
3. 조합 두 가지: **(가) 평균**, **(나) 최댓값**. cal-L 8토큰 원래 정답 전체 탐지율이 높은 쪽을 고르고, 차이가 0.01 미만이면 평균.
4. 임계값은 cal-L 등각 서비스 10% (과제 구분 없이 하나 — 서비스에서는 과제를 모른다).

## 사전 정의 기준 (결과 보기 전, SSOT §6 I3에 고정)

- **I3-1 (주, 약점)**: cal-L 8토큰 원래 정답 **Summary** 탐지율 − F1 base ≥ +0.05, 그리고 응답 단위 쌍체 부트스트랩 95% 하한 > 0
- **I3-2 (전체)**: cal-L 8토큰 원래 정답 전체 탐지율 차이 ≥ 0.00
- **I3-3 (안전)**: C1 전체 차이 ≥ −0.01, Data2txt 차이 ≥ −0.03, 원래 정답 지연 중앙값 증가 ≤ 2토큰
- **I3-4 (지연)**: 두 판정기를 함께 돌린 판정 1회 p95 ≤ 68ms(A100) — 이 노트북에서는 재지 않는다. 측정 전까지 "조건부 채택"
- 채택 = I3-1·2·3 충족 + I3-4 측정 통과. dev-L은 보고만.

> **런타임: CPU로 충분**(약 10~20분, 대부분 데이터 준비). 드라이브에 `r0/`, `r4/`, `r4b/labels_train.json`, `final/scores_base.npy`, `i2/scores_i2.npy`. 결과는 드라이브 `Colab Notebooks/HalluGate/i3/`와 `i3_results.zip`.


## 1. 환경 준비

In [ ]:
!pip -q install pysbd tiktoken
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'

import json, time, random, os, gc, copy
import numpy as np, pandas as pd, torch, pysbd, tiktoken
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/Colab Notebooks/HalluGate' if USE_DRIVE else '.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
STUDENT_DIR = os.path.join(OUT_DIR, 'student_vark')
R0_DIR = os.path.join(OUT_DIR, 'r0')
R1_DIR = I3_DIR = os.path.join(OUT_DIR, 'i3')
FINAL_DIR = os.path.join(OUT_DIR, 'final')
I2_DIR = os.path.join(OUT_DIR, 'i2')
R1B_DIR, R1C_DIR, R4_DIR, R4B_DIR = (os.path.join(OUT_DIR, d) for d in ['r1b', 'r1c', 'r4', 'r4b'])
os.makedirs(R1_DIR, exist_ok=True)
for p in [os.path.join(R4B_DIR, 'labels_train.json'), os.path.join(R4_DIR, 'r4_corrected_labels.json')]:
    assert os.path.exists(p), f'{p} 없음'
for p in [os.path.join(FINAL_DIR, 'scores_base.npy'), os.path.join(I2_DIR, 'scores_i2.npy')]:
    assert os.path.exists(p), f'{p} 없음 (F1 재실행·I2 저장 점수)'
import shutil
print('device', DEVICE, '(GPU 필요 없음)')

## 2. 데이터와 평가 세트 (F1·X1과 같음)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']          # test 분할은 사용하지 않음
random.seed(SEED)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)
dev_ids = {r['id'] for r in dev}
rest = [r for r in train_pool if r['id'] not in dev_ids]
rng = random.Random(11)
teacher_resp = []
for tt in TASKS:
    teacher_resp += rng.sample([r for r in rest if r['task_type'] == tt], 200)
rng.shuffle(teacher_resp)
val = teacher_resp[:int(len(teacher_resp) * 0.15)]                   # student 학습에 쓰지 않은 검증 분할
assert not (dev_ids & {r['id'] for r in val})
print(f'dev {len(dev)} 응답 | 검증 {len(val)} 응답')

In [ ]:
ENC = tiktoken.get_encoding('cl100k_base')
seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def tokenize_offsets(text):
    ids = ENC.encode(text)
    dec, starts = ENC.decode_with_offsets(ids)
    if dec != text:                       # 드문 경우: 멀티바이트 문자 경계 — 누적 디코드로 대체
        starts, pos = [], 0
        for i in range(len(ids)):
            starts.append(pos); pos = len(ENC.decode(ids[:i + 1]))
    ends = starts[1:] + [len(text)]
    return starts, ends

def prep(r):
    text = r['response']
    ts, te = tokenize_offsets(text)
    # 토큰의 '내용 시작' 위치: cl100k 토큰은 앞 공백을 포함하므로(" The"), 공백을 건너뛴 위치로 문장·환각 구간에 배정한다.
    # (v1에서는 공백 위치로 배정해 문장 경계 토큰이 앞 문장에 붙는 오류가 발생 — 발생 위치의 18%)
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) for a, b in zip(ts, te)]
    cs = [c if c < b else a for c, a, b in zip(cs, ts, te)]          # 공백뿐인 토큰
    spans = [(l['start'], l['end']) for l in (r.get('labels') or [])]
    bad = [j for j in range(len(ts)) if any(min(te[j], b) > max(cs[j], a) for a, b in spans)]
    sents = []
    for sp in seg.segment(text):
        if not sp.sent.strip(): continue
        a, b = sp.start, sp.end
        tok = [j for j in range(len(ts)) if a <= cs[j] < b]
        if not tok: continue
        y = int(any(min(b, e) > max(a, s) for s, e in spans))
        sents.append({'a': a, 'b': b, 't0': tok[0], 't1': tok[-1], 'y': y})
    sent_of = {t: sn['t0'] for sn in sents for t in range(sn['t0'], sn['t1'] + 1)}
    return {'id': str(r['id']), 'task_type': r['task_type'], 'gen_model': r['model'],
            'ref_key': r['source_id'], 'text': text, 'ts': ts, 'te': te, 'n_tok': len(ts),
            'onset': bad[0] if bad else None, 'sents': sents, 'sent_of': sent_of,
            'ends': {sn['t1'] for sn in sents}}

REFS = {}
def to_items(rs):
    out = []
    for r in rs:
        REFS.setdefault(r['source_id'], build_reference(sources[r['source_id']]))
        out.append(prep(r))
    return out

dev_items, val_items = to_items(dev), to_items(val)
for name, it in [('dev', dev_items), ('검증', val_items)]:
    n_h = sum(x['onset'] is not None for x in it)
    print(f'{name}: 응답 {len(it)} (환각 {n_h}, 없음 {len(it)-n_h}) | 토큰 {sum(x["n_tok"] for x in it):,} | '
          f'환각 없는 응답 토큰 {sum(x["n_tok"] for x in it if x["onset"] is None):,}')

In [ ]:
# R0 평가 세트(cal-L, dev-L)와 F1 base 저장 점수. 응답 본문·RAGTruth 원 프롬프트는 train 분할에서 찾는다(test는 열지 않음)
import pickle
R0P = pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))
Z0 = dict(np.load(os.path.join(R0_DIR, 'r0_scores.npz')))
CAL, DEV = R0P['items']['calL'], R0P['items']['devL']
for k in {x['ref_key'] for x in CAL + DEV}:
    REFS.setdefault(k, build_reference(sources[k]))
_tr = {str(r['id']): r for r in responses if r['split'] == 'train'}
assert all(x['id'] in _tr for x in CAL + DEV), 'cal-L·dev-L 응답이 train 분할에 없음'
RESP_TEXT = {x['id']: _tr[x['id']]['response'] for x in CAL + DEV}
PROMPT = {x['id']: sources[_tr[x['id']]['source_id']]['prompt'] for x in CAL + DEV}
print(f"cal-L {len(CAL)} 응답 (환각 {sum(x['onset'] is not None for x in CAL)}) | dev-L {len(DEV)} 응답 (환각 {sum(x['onset'] is not None for x in DEV)})")

In [ ]:
# R4 보정 정답 C1 (Opus 재판정)을 cal-L·dev-L에 적용한 사본
CORR = json.load(open(os.path.join(R4_DIR, 'r4_corrected_labels.json')))['C1']
def apply_corr(items, split):
    out = []
    for x in items:
        c = CORR[split].get(x['id'])
        y = copy.copy(x); y['sents'] = [dict(sn) for sn in x['sents']]
        if c is not None:
            y['onset'] = c['onset']
            for i in c['added']: y['sents'][i - 1]['y'] = 1
        out.append(y)
    return out
CAL_C, DEV_C = apply_corr(CAL, 'calL'), apply_corr(DEV, 'devL')
LAB_SETS = {'orig': (CAL, DEV), 'C1': (CAL_C, DEV_C)}
for k, (c, d) in LAB_SETS.items():
    print(f"{k}: cal-L 환각 {sum(x['onset'] is not None for x in c)}, dev-L 환각 {sum(x['onset'] is not None for x in d)}")

## 3. 평가 함수 (F1·I2와 같음)

In [ ]:
SC = dict(Z0)

def series(x, scorer):
    m, k = scorer.split('|')
    S_ = SC[m]
    if k == 'end':
        return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends']]
    if k == 'ew2':
        out, s = [], None
        for t, q in x['pts2']:
            v = float(S_[q]); s = v if s is None else 0.5 * v + 0.5 * s
            out.append((t, s))
        return out
    s = int(k)
    return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends'] or (t - x['sent_of'][t] + 1) % s == 0]

In [ ]:
def calibrate(items, scorer, arl0):
    clean = [x for x in items if x['onset'] is None]
    n_tok = sum(x['n_tok'] for x in clean)
    sc = np.sort(np.concatenate([[s for _, s in series(x, scorer)] for x in clean]))[::-1]
    n_alarm = max(1, int(n_tok / arl0))
    return float(sc[n_alarm - 1]) + 1e-6 if n_alarm <= len(sc) else float(sc[-1])

def evaluate(items, scorer, thr):
    # 엄격 기준(주 지표, 선행 연구와 같은 정의): 응답의 **첫 경보**가 발생 θ 이후여야 탐지로 인정.
    #   θ 전에 경보가 울리면 그 응답은 오경보로 끝난 것(미탐지)으로 본다.
    # 관대 기준(참고): θ 전 경보는 무시하고 θ 이후 첫 경보를 탐지로 인정 — v1의 주 지표였으나 재현율이 부풀려짐.
    clean = [x for x in items if x['onset'] is None]
    hall = [x for x in items if x['onset'] is not None]
    alarms = sum(sum(s >= thr for _, s in series(x, scorer)) for x in clean)
    arl = sum(x['n_tok'] for x in clean) / max(alarms, 1)
    clean_flag = np.mean([any(s >= thr for _, s in series(x, scorer)) for x in clean])
    st_det, st_d, st_ch, cedd, le_det, pre = 0, [], [], [], 0, 0
    for x in hall:
        ser = series(x, scorer)
        first = next((t for t, s in ser if s >= thr), None)
        le_hit = next((t for t, s in ser if t >= x['onset'] and s >= thr), None)
        le_det += le_hit is not None
        pre += first is not None and first < x['onset']
        if first is not None and first >= x['onset']:
            st_det += 1; st_d.append(first - x['onset']); cedd.append(first - x['onset'])
            st_ch.append(x['te'][first] - x['ts'][x['onset']])
        else:
            cedd.append(x['n_tok'] - x['onset'])
    return {'realized_ARL0_dev': arl, 'clean_resp_flagged': clean_flag,
            'recall': st_det / len(hall), 'delay_tok_mean': np.mean(st_d) if st_d else np.nan,
            'delay_tok_median': np.median(st_d) if st_d else np.nan,
            'delay_chars_median': np.median(st_ch) if st_ch else np.nan,
            'censored_EDD_tok': np.mean(cedd), 'pre_onset_alarm_rate': pre / len(hall),
            'recall_lenient': le_det / len(hall),
            'calls_per_resp': np.mean([len(series(x, scorer)) for x in items])}

def calibrate_flag(items, scorer, rate):
    # 서비스 기준 보정: 환각 없는 응답 중 표시되는 비율이 rate 이하가 되는 임계값 (응답별 최대 점수의 분위수)
    mx = np.array([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    return float(np.quantile(mx, 1 - rate)) + 1e-6

In [ ]:
ALPHA = 0.10
def conformal_flag(items, scorer, alpha=ALPHA):
    mx = np.sort([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    n = len(mx); k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(mx[k - 1]) + 1e-6 if k <= n else float('inf')

def detect_vec(items, scorer, thr):
    # 환각 응답마다 엄격 탐지 여부(첫 경보가 발생 이후)와 지연
    det, dl = [], []
    for x in items:
        if x['onset'] is None: continue
        first = next((t for t, s in series(x, scorer) if s >= thr), None)
        ok = first is not None and first >= x['onset']
        det.append(ok); dl.append(first - x['onset'] if ok else np.nan)
    return np.array(det, dtype=float), np.array(dl)

## 4. 두 판정기 점수를 같은 지점 번호로

I2 노트북과 같은 함수로 평가 지점 번호를 다시 만들고, I2 저장 점수 길이와 맞는지, F1 base·I2의 기존 cal-L 8토큰 결과가 재현되는지 확인한다.

In [ ]:
# I2 판정 입력 = (근거 전체, 응답 처음부터 판정 지점 토큰까지). 판정 지점·정답은 F1과 같다.
MAX_LEN2 = 2048                    # L2: RAGTruth train 근거+응답 p90 1,182 → 2,048
N_TRAIN = 80000                    # 학습 지점 표본 (F1은 모든 지점 약 25만 개를 1에폭 — 긴 입력이라 일부만)

def need_point(x, t):
    # 평가에 쓰는 지점만 채점: 문장 끝 + 4토큰 간격 (8·16토큰 간격은 그 부분집합)
    return t in x['ends'] or (t - x['sent_of'][t] + 1) % 4 == 0

def build_eval_points(items, resp_text):
    # 평가 지점마다 새 번호. R0 번호는 (근거, 문장 앞부분)으로 묶여 응답 앞부분이 다른 지점을 구분하지 못한다
    keys, texts, need, MAP = [], [], [], {}
    for x in items:
        text = resp_text[x['id']]
        assert len(text) >= x['te'][-1], x['id']
        for t, _ in x['pts2']:
            MAP[(x['id'], t)] = len(keys)
            keys.append(x['ref_key']); texts.append(text[:x['te'][t]]); need.append(need_point(x, t))
    return keys, texts, np.array(need), MAP

def remap(items, MAP):
    out = []
    for x in items:
        y = copy.copy(x); y['pts2'] = [(t, MAP[(x['id'], t)]) for t, _ in x['pts2']]
        out.append(y)
    return out


def to_E(items, name, MAP, n):
    # 기존 질의 번호 점수(F1 base)를 I2 평가 지점 번호로 옮긴다
    a = np.full(n, np.nan, dtype=np.float32)
    for x in items:
        for t, q in x['pts2']:
            if (x['id'], t) in MAP: a[MAP[(x['id'], t)]] = SC[name][q]
    return a

def clean_stats(items, scorer):
    # 표준화 기준: 멀쩡한 응답(cal-L 원래 정답)의 판정 지점 점수 평균·표준편차
    v = np.concatenate([[s for _, s in series(x, scorer)] for x in items if x['onset'] is None])
    return float(np.mean(v)), float(np.std(v))

def compare_e(cal, a, b, kind='8', task=None, n_boot=2000, seed=2027):
    # a(후보) − b(기준). 임계값은 cal 전체로 등각 서비스 10%(서비스와 같은 단일 임계값), task가 있으면 그 과제 환각 응답만 비교
    ta, tb = conformal_flag(cal, f'{a}|{kind}'), conformal_flag(cal, f'{b}|{kind}')
    sub = [x for x in cal if task is None or x['task_type'] == task]
    da, la = detect_vec(sub, f'{a}|{kind}', ta); db, lb = detect_vec(sub, f'{b}|{kind}', tb)
    rng = np.random.default_rng(seed); n = len(da)
    bt = [da[i].mean() - db[i].mean() for i in (rng.integers(0, n, n) for _ in range(n_boot))]
    return {'task': task or 'all', 'recall_new': float(da.mean()), 'recall_base': float(db.mean()), 'delta': float(da.mean() - db.mean()),
            'ci_lo': float(np.percentile(bt, 2.5)), 'ci_hi': float(np.percentile(bt, 97.5)),
            'delay_new': float(np.nanmedian(la)) if np.isfinite(la).any() else np.nan,
            'delay_base': float(np.nanmedian(lb)) if np.isfinite(lb).any() else np.nan,
            'n_hall': n, 'thr_new': ta, 'thr_base': tb}

def report(cal, dev, name, kind='8'):
    # dev-L 보고: cal 임계값으로 과제별 탐지율·지연과 멀쩡한 응답 표시율
    thr = conformal_flag(cal, f'{name}|{kind}'); out = []
    for task in [None] + TASKS:
        sub = [x for x in dev if task is None or x['task_type'] == task]
        det, dl = detect_vec(sub, f'{name}|{kind}', thr)
        flag = np.mean([any(s >= thr for _, s in series(x, f'{name}|{kind}')) for x in sub if x['onset'] is None])
        out.append({'model': name, 'task': task or 'all', 'recall': float(det.mean()),
                    'delay_med': float(np.nanmedian(dl)) if np.isfinite(dl).any() else np.nan, 'clean_flagged': float(flag)})
    return out

SC['F_base'] = np.load(os.path.join(FINAL_DIR, 'scores_base.npy'))
E_KEYS, E_TEXTS, E_NEED, E_MAP = build_eval_points(CAL + DEV, RESP_TEXT)
SC['I2'] = np.load(os.path.join(I2_DIR, 'scores_i2.npy'))
assert len(SC['I2']) == len(E_KEYS), f'I2 점수 길이 {len(SC["I2"])} ≠ 평가 지점 {len(E_KEYS)} — 지점 번호가 I2 노트북과 다름'
SC['Fb'] = to_E(CAL + DEV, 'F_base', E_MAP, len(E_KEYS))
LS = {lab: (remap(c, E_MAP), remap(d, E_MAP)) for lab, (c, d) in LAB_SETS.items()}
_e8 = [e for x in LS['orig'][0] + LS['orig'][1] for t, e in x['pts2'] if t in x['ends'] or (t - x['sent_of'][t] + 1) % 8 == 0]
assert np.isfinite(SC['Fb'][_e8]).all() and np.isfinite(SC['I2'][_e8]).all(), '8토큰 지점에 점수가 없는 곳이 있음'

CALo, CALc = LS['orig'][0], LS['C1'][0]
_r = compare_e(CALo, 'I2', 'Fb')
print(f"재현 확인 cal-L 8토큰 원래 정답: F1 base {_r['recall_base']:.4f} (임계값 {_r['thr_base']:.4f}) | I2 {_r['recall_new']:.4f} (임계값 {_r['thr_new']:.4f})")
assert abs(_r['thr_base'] - 3.968751) < 1e-3 and abs(_r['recall_base'] - 0.5068) < 2e-3, 'F1 base 재현 실패'
assert abs(_r['thr_new'] - 4.562501) < 1e-3 and abs(_r['recall_new'] - 0.5068) < 2e-3, 'I2 재현 실패 (I2 노트북 10절과 다름)'
print(f'평가 지점 {len(E_KEYS):,}개, 8토큰 지점 {len(set(_e8)):,}개')

## 5. 표준화와 조합, 선택 (cal-L만)

In [ ]:
STATS = {m: clean_stats(CALo, f'{m}|8') for m in ['Fb', 'I2']}
Z = {m: (SC[m] - mu) / sd for m, (mu, sd) in STATS.items()}
SC['MEAN'] = (Z['Fb'] + Z['I2']) / 2
SC['MAX'] = np.maximum(Z['Fb'], Z['I2'])
print('표준화 기준 (평균, 표준편차):', {m: tuple(round(v, 4) for v in s) for m, s in STATS.items()})

SEL_ROWS = [{'combo': c, **compare_e(CALo, c, 'Fb')} for c in ['MEAN', 'MAX']]
sel_tab = pd.DataFrame(SEL_ROWS)
print(sel_tab[['combo', 'recall_new', 'recall_base', 'delta', 'ci_lo', 'ci_hi', 'delay_new', 'delay_base']].round(3).to_string(index=False))
r_mean, r_max = SEL_ROWS[0]['recall_new'], SEL_ROWS[1]['recall_new']
CHOSEN = 'MAX' if r_max - r_mean >= 0.01 else 'MEAN'
print(f'=== 선택: {CHOSEN} (평균 {r_mean:.3f}, 최댓값 {r_max:.3f}) ===')

## 6. 사전 기준 판정 (cal-L)

In [ ]:
ROWS = []
for lab, cal in [('orig', CALo), ('C1', CALc)]:
    for task in [None] + TASKS:
        for m in [CHOSEN, 'I2']:
            ROWS.append({'labels': lab, 'model': m, **compare_e(cal, m, 'Fb', task=task)})
cal_tab = pd.DataFrame(ROWS)
pd.set_option('display.width', 250)
print(cal_tab[['labels', 'model', 'task', 'recall_new', 'recall_base', 'delta', 'ci_lo', 'ci_hi', 'delay_new', 'delay_base', 'n_hall']].round(3).to_string(index=False))
def row(lab, task, m=None):
    return next(r for r in ROWS if r['labels'] == lab and r['task'] == (task or 'all') and r['model'] == (m or CHOSEN))
s, a, c, d = row('orig', 'Summary'), row('orig', None), row('C1', None), row('orig', 'Data2txt')
I3_1 = s['delta'] >= 0.05 and s['ci_lo'] > 0
I3_2 = a['delta'] >= 0.0
I3_3 = c['delta'] >= -0.01 and d['delta'] >= -0.03 and (a['delay_new'] - a['delay_base']) <= 2
print(f"=== I3-1 (주) Summary {s['recall_base']:.3f} → {s['recall_new']:.3f} (Δ {s['delta']:+.3f} [{s['ci_lo']:+.3f}, {s['ci_hi']:+.3f}], 환각 {s['n_hall']}) → {'충족' if I3_1 else '미달'} ===")
print(f"=== I3-2 (전체) {a['recall_base']:.3f} → {a['recall_new']:.3f} (Δ {a['delta']:+.3f} [{a['ci_lo']:+.3f}, {a['ci_hi']:+.3f}]) → {'충족' if I3_2 else '미달'} ===")
print(f"=== I3-3 (안전) C1 Δ {c['delta']:+.3f}, Data2txt Δ {d['delta']:+.3f}, 지연 {a['delay_base']:.0f} → {a['delay_new']:.0f} → {'충족' if I3_3 else '미달'} ===")
ADOPT_COND = bool(I3_1 and I3_2 and I3_3)
print(f"=== I3: {'조건부 채택 (I3-4 지연 측정 필요)' if ADOPT_COND else '미채택'} ===")

## 7. dev-L 확인 (보고만)

In [ ]:
DEV_ROWS = []
for lab in ['orig', 'C1']:
    cal, dev = LS[lab]
    for m in ['Fb', 'I2', 'MEAN', 'MAX']:
        DEV_ROWS += [{'labels': lab, **r} for r in report(cal, dev, m)]
dev_tab = pd.DataFrame(DEV_ROWS)
print(dev_tab.round(3).to_string(index=False))

## 8. 저장과 zip 다운로드

In [ ]:
LOCAL = '/content/i3_local'; os.makedirs(LOCAL, exist_ok=True)
sel_tab.to_csv(os.path.join(LOCAL, 'i3_select.csv'), index=False)
cal_tab.to_csv(os.path.join(LOCAL, 'i3_calL.csv'), index=False)
dev_tab.to_csv(os.path.join(LOCAL, 'i3_devL.csv'), index=False)
summ = {'chosen': CHOSEN, 'I3_1': bool(I3_1), 'I3_2': bool(I3_2), 'I3_3': bool(I3_3), 'adopt_conditional': ADOPT_COND,
        'standardize': {m: {'mean': mu, 'std': sd} for m, (mu, sd) in STATS.items()},
        'threshold_svc10_8': conformal_flag(CALo, f'{CHOSEN}|8'), 'summary': s, 'all': a, 'C1': c, 'Data2txt': d}
json.dump(summ, open(os.path.join(LOCAL, 'i3_summary.json'), 'w'), ensure_ascii=False, indent=1, default=float)
for f in os.listdir(LOCAL):
    try: shutil.copy(os.path.join(LOCAL, f), os.path.join(I3_DIR, f))
    except OSError as e: print('드라이브 저장 실패', f, e)
zp = shutil.make_archive('/content/i3_results', 'zip', LOCAL)
from google.colab import files
files.download(zp)
print(json.dumps({k: summ[k] for k in ['chosen', 'I3_1', 'I3_2', 'I3_3', 'adopt_conditional']}, ensure_ascii=False))